# 00 学习路线：先从问题而不是目录开始

这个目录是为了学习项目而建的，不是训练产物目录。你可以放心打开运行：默认只读项目代码和数据，不覆盖 `data/`、`scripts/`、`outputs/`。

这一套 notebook 不再按流水账讲“项目有啥文件”，而是按面试和真实理解中最关键的问题来拆：

1. Qwen 是怎么下载、缓存、加载并完成一次回答的？
2. tokenizer 和 chat template 为什么是大模型训练的入口？
3. SFT 数据怎么从 JSONL 变成 token，loss 又到底算在哪里？
4. LoRA 在代码里怎么贴到 Qwen 上，为什么它省显存？
5. adapter 是怎么保存、加载和对比的？
6. DPO 的 chosen/rejected 是怎么进入训练的？
7. 最后为什么不盲目接受 DPO，而是用行为门禁验收？

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, textwrap, re

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts").exists():
    PROJECT_ROOT = Path(r"D:/coding/qwen lorar sft/qwen-local-lora-sft-dpo")

print("项目根目录:", PROJECT_ROOT)
print("学习原则: 本 notebook 默认只读项目文件；所有真实推理/训练开关默认 False。")

def read_text(rel):
    return (PROJECT_ROOT / rel).read_text(encoding="utf-8", errors="replace")

def show_file(rel, start=1, end=None):
    lines = read_text(rel).splitlines()
    if end is None:
        end = len(lines)
    print(f"--- {rel}:{start}-{end} ---")
    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:03d}: {lines[i-1]}")

def find_lines(rel, keyword, context=4):
    lines = read_text(rel).splitlines()
    hits = []
    for idx, line in enumerate(lines, start=1):
        if keyword in line:
            hits.append(idx)
    if not hits:
        print("没有找到:", keyword)
        return
    for hit in hits:
        start = max(1, hit - context)
        end = min(len(lines), hit + context)
        show_file(rel, start, end)
        print()

def read_jsonl(rel, n=3):
    rows = []
    with (PROJECT_ROOT / rel).open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
            if len(rows) >= n:
                break
    return rows

def count_jsonl(rel):
    with (PROJECT_ROOT / rel).open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

## 1. 先确认学习目录不干扰项目

下面这格只检查目录和关键文件是否存在。它不会写任何文件。

In [ ]:
for rel in [
    "scripts/infer.py",
    "scripts/train_sft_lora.py",
    "scripts/train_dpo.py",
    "data/processed/custom_sft_train.jsonl",
    "data/processed/dpo_tiny_train.jsonl",
    "outputs/sft_lora_qwen05b_custom_v3_from_v1_patch",
    "outputs/dpo_lora_qwen05b_naive_v6",
]:
    p = PROJECT_ROOT / rel
    print(f"{rel:60s}", "OK" if p.exists() else "MISSING")

## 2. 这个项目的一句话底层模型

你可以先把项目想成三个层次：

```text
底座模型 Qwen2.5-0.5B-Instruct
  + LoRA adapter 小参数补丁
  + 行为评估门禁，看回答是否真的变好
```

SFT 负责“模仿标准答案”。DPO 负责“偏向 chosen，远离 rejected”。LoRA 负责“不要改整本大模型，只训练小补丁”。

## 3. 推荐学习顺序

- 01 先看 Qwen 怎么加载和回答。
- 02 再看 tokenizer 怎么把聊天变成 token。
- 03 看 SFT 样本和 loss。
- 04 看 LoRA 怎么在代码里实现。
- 05 看 adapter 怎么保存、加载、对比。
- 06 看 DPO 的 preference 数据和训练入口。
- 07 把所有东西组织成面试能讲的工程闭环。